### FASE 3: División de Datos 

| Vamos a extraer caracteristicas por cada registro

Importamos librerias

In [3]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
tqdm.pandas()
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit,cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pyswarms as ps
from pyswarms.utils.functions import single_obj as fx
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('completo_clusters_con_features.csv')

df_model = df.copy()

le = LabelEncoder()
df_model['Direccion'] = le.fit_transform(df_model['Direccion'])


features = [
    'Total_Vehiculos','Tiempo_Medio_s', 'Ocupacion_Espacial_%', 
    'Hora_Minutos', 'Dia_Semana', 'Direccion', 'Total_Vehiculos_lag1', 'Ocupacion_lag1', 
    'Media_Movil_3ciclos', 'Tendencia_Vehiculos', 'Saturacion_Actual',
    'Periodo_Dia',  
]


entreno_data = df_model[df_model['Dia_Semana'].isin([1, 2,5])]
val_data   = df_model[df_model['Dia_Semana'] == 3]
prueba_data  = df_model[df_model['Dia_Semana'] == 4]


X_entreno, y_entreno = entreno_data[features], entreno_data['Tiempo_Optimo']
X_val, y_val     = val_data[features], val_data['Tiempo_Optimo']
X_prueba, y_prueba   = prueba_data[features], prueba_data['Tiempo_Optimo']

df_model = df_model.sort_values(by=['Dia_Semana', 'Hora_Minutos']).reset_index(drop=True)

X = df_model[features]
y = df_model['Tiempo_Optimo']


df_model  = pd.concat([entreno_data, val_data, prueba_data])


### FASE 4: Optimización con PSO

In [8]:

def pso_rf_optimization_robust(X, y, n_particles=12, n_iters=10):

    MAX_FEAT_OPTS = ['sqrt', 'log2', 1.0] 

    def fitness_function(particles):

        scores = []
        
        for particle in particles:
            n_estimators = int(particle[0])
            max_depth = int(particle[1])
            min_samples_split = int(particle[2])
            min_samples_leaf = int(particle[3])
            
            feat_idx = int(particle[4])
            if feat_idx >= len(MAX_FEAT_OPTS): feat_idx = len(MAX_FEAT_OPTS) - 1
            max_features = MAX_FEAT_OPTS[feat_idx]
            
            final_depth = max_depth if max_depth > 1 else None
            rf = RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=final_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                random_state=42,
                n_jobs=-1 
            )
            
            try:
                cv_scores = cross_val_score(rf, X, y, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
                mean_mse = -1 * np.mean(cv_scores)
                scores.append(mean_mse)
            except Exception as e:
                scores.append(1e5) 
        
        return np.array(scores)
    lb = [100,  2, 2,  1, 0]    # Mínimos (Más árboles = más estabilidad)
    ub = [1000, 50, 20, 10, 2.99] # Máximos (Índice features llega casi a 3)
    bounds = (lb, ub)
    
    options = {'c1': 0.6, 'c2': 0.4, 'w': 0.8} 
    
    optimizer = ps.single.GlobalBestPSO(
        n_particles=n_particles, 
        dimensions=5,
        options=options,
        bounds=bounds
    )
        
    best_cost, best_pos = optimizer.optimize(fitness_function, iters=n_iters)
    
    final_feat_idx = int(best_pos[4])
    if final_feat_idx >= len(MAX_FEAT_OPTS): final_feat_idx = len(MAX_FEAT_OPTS) - 1
    
    best_params = {
        'n_estimators': int(best_pos[0]),
        'max_depth': int(best_pos[1]) if int(best_pos[1]) > 1 else None,
        'min_samples_split': int(best_pos[2]),
        'min_samples_leaf': int(best_pos[3]),
        'max_features': MAX_FEAT_OPTS[final_feat_idx]
    }
    
    return best_params, best_cost

best_params, best_cost = pso_rf_optimization_robust(X, y)

print("\n=== MEJORES HIPERPARÁMETROS (Validación Cruzada) ===")
print(best_params)
print(f"Mejor MSE promedio (CV): {best_cost:.4f}")
print(f"RMSE estimado: {np.sqrt(best_cost):.4f}")

2026-01-29 14:34:53,071 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.6, 'c2': 0.4, 'w': 0.8}
pyswarms.single.global_best:   0%|          |0/10


KeyboardInterrupt: 

### FASE 5: Modelo Random Forest Optimizado

In [16]:

tscv = TimeSeriesSplit(n_splits=4)

print(f"Iniciando Validación Cruzada con Expansión (Total muestras: {len(X)})")

mae_scores = []
mse_scores = []
rmse_scores = []
r2_scores = []

for i, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

# Esta configuracion coloque fijo a lo que dice la documentacion .md, esta se optubo de la face anterior
    rf_fold = RandomForestRegressor(

        n_estimators=456, max_depth=7, min_samples_split=5, min_samples_leaf=3, max_features=1.0,
        random_state=42,
        n_jobs=-1
    )
    rf_fold.fit(X_train_fold, y_train_fold)
    
    y_pred_fold = rf_fold.predict(X_test_fold)
    
    mae = mean_absolute_error(y_test_fold, y_pred_fold)
    mse = mean_squared_error(y_test_fold, y_pred_fold)
    rmse = np.sqrt(mean_squared_error(y_test_fold, y_pred_fold))
    r2 = r2_score(y_test_fold, y_pred_fold)
    
    mae_scores.append(mae)
    mse_scores.append(mse)
    rmse_scores.append(rmse)
    r2_scores.append(r2)
    
    print(f"Iteración {i+1}: Train tam={len(train_index)} | Test tam={len(test_index)} -> MAE: {mae:.3f} | R²: {r2:.4f}")



Iniciando Validación Cruzada con Expansión (Total muestras: 3801)
Iteración 1: Train tam=761 | Test tam=760 -> MAE: 1.102 | R²: 0.8865
Iteración 2: Train tam=1521 | Test tam=760 -> MAE: 1.295 | R²: 0.7931
Iteración 3: Train tam=2281 | Test tam=760 -> MAE: 1.142 | R²: 0.8640
Iteración 4: Train tam=3041 | Test tam=760 -> MAE: 1.146 | R²: 0.8695


### FASE 6: Evaluación Final 


In [6]:
print("\n=== RESULTADOS PROMEDIO (VALIDACIÓN ROBUSTA) ===")
print(f"MAE Promedio:  {np.mean(mae_scores):.3f}")
print(f"MSE Promedio:  {np.mean(mse_scores):.3f}")
print(f"RMSE Promedio: {np.mean(rmse_scores):.3f}")
print(f"R² Promedio:   {np.mean(r2_scores):.4f}")

rf_final = RandomForestRegressor(n_estimators=456, max_depth=7, min_samples_split=5, min_samples_leaf=3, max_features=1.0, random_state=42, n_jobs=-1)
rf_final.fit(X, y)
print("\n✅ Modelo final entrenado con el 100% de la historia disponible.")



=== RESULTADOS PROMEDIO (VALIDACIÓN ROBUSTA) ===
MAE Promedio:  1.149
MSE Promedio:  7.704
RMSE Promedio: 2.762
R² Promedio:   0.8603

✅ Modelo final entrenado con el 100% de la historia disponible.


### FASE 7: Prediccion de tiempos verdes

In [7]:
X_prueba = prueba_data[X.columns]
y_prueba_pred = rf_final.predict(X_prueba)

def comparar_vs_tiempo_fijo(test_data, y_test_pred, fijo_verde=30, amarillo=3):
    df = test_data.copy()
    
    df['Verde_Modelo'] = y_test_pred
    df['Verde_Fijo'] = fijo_verde
    df['Diferencia'] = df['Verde_Modelo'] - df['Verde_Fijo']

    resultados = {}

    for direccion in sorted(df['Direccion'].unique()):
        datos = df[df['Direccion'] == direccion]

        promedio_modelo = datos['Verde_Modelo'].mean()
        promedio_fijo = fijo_verde

        resultados[f'Dir_{direccion}'] = round(promedio_modelo)

        print(f"\nDIRECCIÓN {direccion + 1}")
        print(f"Modelo ML : {promedio_modelo:.1f} s")
        print(f"Fijo      : {promedio_fijo:.1f} s")
        print(f"Diferencia: {promedio_modelo - promedio_fijo:+.1f} s")

    return resultados
resultados_comparativa = comparar_vs_tiempo_fijo(
    prueba_data,
    y_prueba_pred,
    fijo_verde=30,
    amarillo=3
)



DIRECCIÓN 1
Modelo ML : 26.0 s
Fijo      : 30.0 s
Diferencia: -4.0 s

DIRECCIÓN 2
Modelo ML : 26.8 s
Fijo      : 30.0 s
Diferencia: -3.2 s

DIRECCIÓN 3
Modelo ML : 29.3 s
Fijo      : 30.0 s
Diferencia: -0.7 s

DIRECCIÓN 4
Modelo ML : 28.3 s
Fijo      : 30.0 s
Diferencia: -1.7 s
